# Best encoding settings

The aim here is to train a model on existing data in order to estimate the FFmpeg parameters that optimise any arbitrary criterion.

In [1]:
import pathlib
import subprocess
import tempfile

import cutcutcodec

from mendevi.encode import get_transcode_cmd
from mendevi.models import Solver
from mendevi.models.nn import EnergyNN, QualityNN

In [2]:
dataset = "scatter.db"  # the dataset name, automatically downloaded
val = 0.2  # keep 80% of data for training, 20% for validation
video = cutcutcodec.utils.get_project_root() / "media" / "video" / "intro.webm"

## Train the model

Before making predictions, you first need to train a model.

### Quality model

Quality does not depend on the machine. We therefore produce just one large, unique model.

In [3]:
# declaration
quality_model = QualityNN(
    output_labels=["log_rate", "psnr", "ssim"],
    weights="quality",  # optional, for checkpoint (in ~/.cache/mendevi)
    niter=15000,  # number of training iterations, default 10000
    learning_rate=1e-5,  # adam gradient decay learning rate, default 1e-4
    total=40_000,  # number of model parameters, default based on number of observations
)

In [4]:
# train (on cpu encoders only)
if not quality_model.is_fit():
    quality_model.fit(
        dataset,
        table="t_enc_encode",  # optional, help to find the good SQL request
        select=f"uniform(vid_hash) >= {val} and encoder_family == 'cpu'",
    )

Fit 'Quality estimation based on a neuronal network'...
█ get SQL extractor
█ perform SQL query
█ Fit on cluster ...
█ █ this cluster contains 17853 observations
█ █ Save model into /home/rrichard/.cache/mendevi/quality_29_3_40000.pk.xz...                                        
█ █ █ 143.69 ms
█ █ 58 min 56 s
█ 1 h 1 min
Save model into /home/rrichard/.cache/mendevi/quality.pk.xz...
█ 163.05 ms


In [5]:
# validation
quality_model.validate(dataset, table="t_enc_encode", select=f"uniform(vid_hash) < {val} and encoder_family == 'cpu'")

Validate 'Quality estimation based on a neuronal network'...
█ get SQL extractor
█ perform SQL query
█ Validate on cluster ...
█ █ this cluster contains 4551 observations
█ █ 451.71 ms
█ 1 min 4 s
MAPE of log_rate: 1.12%
RMS  of log_rate: 0.0977
MAPE of psnr: 2.35%
RMS  of psnr: 1.646
MAPE of ssim: 1.30%
RMS  of ssim: 0.01953
MAPE of rate: 16.95%
RMS  of rate: 2.262e+07


### Energy model

Energy depends ont the machine. We therefore produce several models, one per hostname.

In [6]:
energy_model = EnergyNN(
    weights="energy",  # optional, for checkpoint (in ~/.cache/mendevi)
    niter=8000,  # number of training iterations, default 10000
    learning_rate=2e-5,  # adam gradient decay learning rate, default 1e-4
    total=4000,  # number of model parameters, default based on number of observations
)

In [7]:
if not energy_model.is_fit():
    energy_model.fit(
        dataset,
        table="t_enc_encode",  # optional, help to find the good SQL request
        select=f"uniform(vid_hash) >= {val} and encoder_family == 'cpu'",
    )

Fit 'Encoding energy estimation based on a neuronal network'...
█ get SQL extractor
█ perform SQL query
█ Fit on cluster hostname=abacus26-1.rennes.grid5000.fr...
█ █ this cluster contains 6500 observations
█ █ Save model into /home/rrichard/.cache/mendevi/energy_29_2_2000.pk.xz...                                          
█ █ █ 20.35 ms
█ █ 3 min 5 s
█ Fit on cluster hostname=dahu-12.grenoble.grid5000.fr...
█ █ this cluster contains 2539 observations
█ █ Load model from /home/rrichard/.cache/mendevi/energy_29_2_2000.pk.xz...
█ █ █ 25.07 ms
█ █ Save model into /home/rrichard/.cache/mendevi/energy_29_2_2000.pk.xz...                                          
█ █ █ 34.57 ms
█ █ 2 min 2 s
█ Fit on cluster hostname=gemini-2.lyon.grid5000.fr...
█ █ this cluster contains 850 observations
█ █ Load model from /home/rrichard/.cache/mendevi/energy_29_2_2000.pk.xz...
█ █ █ 15.55 ms
█ █ Save model into /home/rrichard/.cache/mendevi/energy_29_2_2000.pk.xz...                                          

In [8]:
# validation
energy_model.validate(dataset, table="t_enc_encode", select=f"uniform(vid_hash) < {val} and encoder_family == 'cpu'")

Validate 'Encoding energy estimation based on a neuronal network'...
█ get SQL extractor
█ perform SQL query
█ Validate on cluster hostname=abacus26-1.rennes.grid5000.fr...
█ █ this cluster contains 1687 observations
█ █ 103.47 ms
█ Validate on cluster hostname=dahu-12.grenoble.grid5000.fr...
█ █ this cluster contains 676 observations
█ █ 52.28 ms
█ Validate on cluster hostname=gemini-2.lyon.grid5000.fr...
█ █ this cluster contains 256 observations
█ █ 27.59 ms
█ Validate on cluster hostname=nova-12.lyon.grid5000.fr...
█ █ this cluster contains 476 observations
█ █ 45.03 ms
█ Validate on cluster hostname=paradoxe-32.rennes.grid5000.fr...
█ █ this cluster contains 618 observations
█ █ 65.60 ms
█ Validate on cluster hostname=suzanne...
█ █ this cluster contains 405 observations
█ █ 49.64 ms
█ Validate on cluster hostname=yeti-2.grenoble.grid5000.fr...
█ █ this cluster contains 433 observations
█ █ 38.65 ms
█ 1 min 23 s
MAPE of log_act_duration_per_frame: 22.78%
RMS  of log_act_duration_p

## Use the model to perform predictions

The idea is to run a huge number of predictions using the trained model in order to explore a very large range of scenarios.

In [9]:
# example best compression for 40 dB

def best_dbrate(**kwd: dict[str]) -> float:
    """Loss function of debit rate distortion around 40 dB."""
    db_loss = 0.0 if 39 < kwd["psnr"] < 41 else ((kwd["psnr"]-40)**2)-1
    rate_loss = kwd["log_rate"]
    return db_loss + 1e-2*rate_loss

solver = Solver(quality_model, best_dbrate)
tested_combinations, loss, cmd = solver.solve(
    video,
    encoder=["libx264", "libx265", "libaom-av1", "libsvtav1"],
    return_cmd=True,
)

Solve...
█ Get properties of intro.webm...
█ █ fps          : 29.97 Hz
█ █ eotf         : bt1361e, bt1361
█ █ pix fmt      : yuv420p
█ █ size         : 0.50 MB
█ █ range        : tv
█ █ gamut        : bt709
█ █ codec        : vp9
█ █ duration     : 9.81 s
█ █ frames       : 294 frames
█ █ width        : 1280 pixels
█ █ name         : intro.webm
█ █ height       : 720 pixels
█ █ rms_sobel    : 0.15 +/- 0.07                                                                                     
█ █ rms_time_diff: 0.08 +/- 0.05
█ █ spatial_dct  : 0.05 +/- 0.03
█ █ temporal_dct : 0.01 +/- 0.01
█ Predict 'Quality estimation based on a neuronal network'...
█ █ check args
█ █ vectorize args
█ █ predict
█ █ Predict for cluster ...
█ █ █ this cluster contains 990 observations
█ █ █ 51.77 ms
█ █ 90.09 ms
█ Result for loss = 0.05546:
█ █ effort       : medium
█ █ encoder      : libx265
█ █ height       : 720
█ █ log_rate     : 5.546
█ █ mode         : crf
█ █ nb_frames    : 294
█ █ psnr         : 39

In [10]:
# perform the conversion
out_video = pathlib.Path(tempfile.gettempdir()) / "video_test.mp4"
cmd.output = [*cmd.output[:-1], str(out_video)]
output, activity = cmd.run()

In [11]:
# final verification
out = subprocess.run(
    ["mendevi", "probe", str(out_video), "-r", str(video), "-d", str(out_video.with_suffix(".db"))],
    check=True,
)

Parse configuration...
█ video     : /tmp/video_test.mp4
█ database  : /tmp/video_test.db (just created)
█ ref       : intro.webm
█ lpips_alex: no
█ lpips_vgg : no
█ psnr      : yes
█ rms_sobel : yes
█ rms_time_diff: yes
█ spatial_dct: yes
█ ssim      : yes
█ temporal_dct: yes
█ uvq       : no
█ vif       : no
█ vmaf      : no
Probe 1/2...                                                     
█ Get properties of intro.webm...
█ █ frames       : 294 frames
█ █ name         : intro.webm
█ █ width        : 1280 pixels
█ █ fps          : 29.97 Hz
█ █ range        : tv
█ █ pix fmt      : yuv420p
█ █ eotf         : bt1361e, bt1361
█ █ duration     : 9.81 s
█ █ size         : 0.50 MB
█ █ codec        : vp9
█ █ height       : 720 pixels
█ █ gamut        : bt709
█ █ rms_sobel    : 0.15 +/- 0.07                                                                                                                                                 
█ █ rms_time_diff: 0.08 +/- 0.05
█ █ spatial_dct  : 0.05 +/